# Vision Agents

## Providing Images at the Start of the Agent’s Execution

In [8]:
!uv add smolagents selenium helium pillow

Resolved 350 packages in 20ms
Audited 319 packages in 90ms


In [9]:
from PIL import Image
import requests
from io import BytesIO

image_urls = [
    "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRWfPULMFLt7DOo7zppITfkFqT87WcZ_N6ZK5OV1AVQBQbGlAGHZt91vsEX&s=10",
    "https://www.roadtoaddis.com/wp-content/uploads/2025/06/Addis-Ababa-Bole-International-airport.jpg"
]

images = []
for url in image_urls:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36" 
    }
    response = requests.get(url,headers=headers)
    image = Image.open(BytesIO(response.content)).convert("RGB")
    images.append(image)

images


[<PIL.Image.Image image mode=RGB size=640x480>,
 <PIL.Image.Image image mode=RGB size=1080x810>]

In [10]:
from smolagents import CodeAgent, OpenAIServerModel

model = OpenAIServerModel(model_id="gpt-4o")

agent = CodeAgent(
    tools=[],
    model=model,
    max_steps=4,
    verbosity_level=2,
    additional_authorized_imports=["PIL"]
)

response = agent.run(
    """
    You have access to a variable named `images`,
    Are they images of an air ticket? If not, what are they?
    """,
    images=images,
)

/Users/nehe/projects/ai_agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You have access to a variable named `images`,                                                                   │
│     Are they images of an air ticket? If not, what are they?                                                    │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o ──────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
The images are not of an air ticket. They depict a view of an airport building.                                    

Error in code parsing:
Your code snippet is invalid, because the regex pattern <code>(.*?)</code> was not found in it.
Here is your code snippet:
The images are not of an air ticket. They depict a view of an airport building.</code>
Make sure to include code with the correct pattern, for instance:
Thoughts: Your thoughts
<code>
# Your python code here
</code>
Make sure to provide correct code blobs.

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I need to identify the content of the images. They appear to be of a building structure, not an air       
ticket.                                                                                                            
                                                                                                                   
<code>                                                                                                             
# Since I cannot identify people or specific textual details, I'll describe the image.                             
description = "The images show an airport building labeled as 'Addis Ababa Bole International Airport.'"           
final_answer(description)                                                                                          
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Since I cannot identify people or specific textual details, I'll describe the image.                           
  description = "The images show an airport building labeled as 'Addis Ababa Bole International Airport.'"         
  final_answer(description)                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The images show an airport building labeled as 'Addis Ababa Bole International Airport.'

[Step 3: Duration 6.06 seconds| Input tokens: 13,590 | Output tokens: 110]

## Providing Images with Dynamic Retreival

In [12]:
from io import BytesIO
from time import sleep

import helium
from dotenv import load_dotenv
from PIL import Image
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

from smolagents import CodeAgent, tool
from smolagents.agents import ActionStep

load_dotenv()

True

In [15]:
@tool
def search_item_ctrl_f(text: str, nth_result: int = 1) -> str:
    """
    Searches for text on the current page via Ctrl + F and jumps to the nth occurrence.
    Args:
        text: The text to search for
        nth_result: Which occurrence to jump to (default: 1)
    """
    elements = driver.find_elements(By.XPATH, f"//*[contains(text(), '{text}')]")
    if nth_result > len(elements):
        raise Exception(f"Match n°{nth_result} not found (only {len(elements)} matches found)")
    result = f"Found {len(elements)} matches for '{text}'."
    elem = elements[nth_result - 1]
    driver.execute_script("arguments[0].scrollIntoView(true);", elem)
    result += f"Focused on element {nth_result} of {len(elements)}"
    return result

@tool
def go_back() -> None:
    """Goes back to previous page."""
    driver.back()

@tool
def close_popups() -> str:
    """
    Closes any visible modal or pop-up on the page. Use this to dismiss pop-up windows!
    This does not work on cookie consent banners.
    """
    webdriver.ActionChains(driver).send_keys(Keys.ESCAPE).perform()

In [13]:
# Configure Chrome options
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--force-device-scale-factor=1")
chrome_options.add_argument("--window-size=1000,1350")
chrome_options.add_argument("--disable-pdf-viewer")
chrome_options.add_argument("--window-position=0,0")

# Initialize the browser
driver = helium.start_chrome(headless=False, options=chrome_options)

# Set up screenshot callback
def save_screenshot(memory_step: ActionStep, agent: CodeAgent) -> None:
    sleep(1.0)  # Let JavaScript animations happen before taking the screenshot
    driver = helium.get_driver()
    current_step = memory_step.step_number
    if driver is not None:
        for previous_memory_step in agent.memory.steps:  # Remove previous screenshots for lean processing
            if isinstance(previous_memory_step, ActionStep) and previous_memory_step.step_number <= current_step - 2:
                previous_memory_step.observations_images = None
        png_bytes = driver.get_screenshot_as_png()
        image = Image.open(BytesIO(png_bytes))
        print(f"Captured a browser screenshot: {image.size} pixels")
        memory_step.observations_images = [image.copy()]  # Create a copy to ensure it persists

    # Update observations with current URL
    url_info = f"Current url: {driver.current_url}"
    memory_step.observations = (
        url_info if memory_step.observations is None else memory_step.observations + "\n" + url_info
    )

In [16]:
# from smolagents import InferenceClientModel
from smolagents import OpenAIServerModel

# model_id = "Qwen/Qwen2-VL-7B-Instruct" 
# model = InferenceClientModel(model_id=model_id)

model = OpenAIServerModel(model_id="gpt-4o")

agent = CodeAgent(
    tools=[go_back, close_popups, search_item_ctrl_f],
    model=model,
    additional_authorized_imports=["helium"],
    step_callbacks=[save_screenshot],
    max_steps=15,
    verbosity_level=2,
)

# Import helium for the agent
agent.python_executor("from helium import *")


CodeOutput(output=None, logs='', is_final_answer=False)

In [18]:
helium_instructions = """
You can use helium to access websites. Don't bother about the helium driver, it's already managed.
We've already ran "from helium import *"
Then you can go to pages!
Code:
```py
go_to('github.com/trending')
```<end_code>

You can directly click clickable elements by inputting the text that appears on them.
Code:
```py
click("Top products")
```<end_code>

If it's a link:
Code:
```py
click(Link("Top products"))
```<end_code>

If you try to interact with an element and it's not found, you'll get a LookupError.
In general stop your action after each button click to see what happens on your screenshot.
Never try to login in a page.

To scroll up or down, use scroll_down or scroll_up with as an argument the number of pixels to scroll from.
Code:
```py
scroll_down(num_pixels=1200) # This will scroll one viewport down
```<end_code>

When you have pop-ups with a cross icon to close, don't try to click the close icon by finding its element or targeting an 'X' element (this most often fails).
Just use your built-in tool `close_popups` to close them:
Code:
```py
close_popups()
```<end_code>
         T
You can use .exists() to check for the existence of an element. For example:
Code:
```py
if Text('Accept cookies?').exists():
    click('I accept')
```<end_code>
"""

In [19]:
url = "https://www.currys.co.uk/products/ninja-foodi-max-15in1-smartlid-ol750uk-multicooker-and-air-fryer-stainless-steel-and-black-10228923.html"
search_request = f"Please navigate to ${url} and find out the price of the item including it's name"

agent_output = agent.run(search_request + helium_instructions)
print("Final output:")
print(agent_output)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Please navigate to                                                                                              │
│ $https://www.currys.co.uk/products/ninja-foodi-max-15in1-smartlid-ol750uk-multicooker-and-air-fryer-stainless-s │
│ teel-and-black-10228923.html and find out the price of the item including it's name                             │
│ You can use helium to access websites. Don't bother about the helium driver, it's already managed.              │
│ We've already ran "from helium import *"                                                                        │
│ Then you can go to pages!                                                                                       │
│ Code:                                                                                                           │
│ ```py                                                                                                           │
│ go_to('github.com/trending')                                                                                    │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ You can directly click clickable elements by inputting the text that appears on them.                           │
│ Code:                                                                                                           │
│ ```py                                                                                                           │
│ click("Top products")                                                                                           │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ If it's a link:                                                                                                 │
│ Code:                                                                                                           │
│ ```py                                                                                                           │
│ click(Link("Top products"))                                                                                     │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ If you try to interact with an element and it's not found, you'll get a LookupError.                            │
│ In general stop your action after each button click to see what happens on your screenshot.                     │
│ Never try to login in a page.                                                                                   │
│                                                                                                                 │
│ To scroll up or down, use scroll_down or scroll_up with as an argument the number of pixels to scroll from.     │
│ Code:                                                                                                           │
│ ```py                                                                                                           │
│ scroll_down(num_pixels=1200) # This will scroll one viewport down                                               │
│ ```<end_code>                                                                                                   │
│                                                       

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
To find out the price and name of the item on the specified webpage, I will navigate to the website using Helium   
and extract the required information by targeting the relevant elements on the page. Let's proceed with these      
steps.                                                                                                             
                                                                                                                   
1. Navigate to the URL for the product page on Currys.                                                             
2. Close any pop-ups if they appear.                                                                               
3. Extract the product name and price.                                                                             
                                                                                                                   
Let's execute the steps:                                                                                           
                                                                                                                   
```python                                                                                                          
# Step 1: Navigate to the URL                                                                                      
go_to('https://www.currys.co.uk/products/ninja-foodi-max-15in1-smartlid-ol750uk-multicooker-and-air-fryer-stainless
-steel-and-black-10228923.html')                                                                                   
```                                                                                                                

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Step 1: Navigate to the URL                                                                                    
  go_to('https://www.currys.co.uk/products/ninja-foodi-max-15in1-smartlid-ol750uk-multicooker-and-air-fryer-stain  
  less-steel-and-black-10228923.html')                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution exceeded the maximum execution time of 30 seconds

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer({                                                                                                   
      "Product Name": "NINJA Foodi MAX 15-in-1 SmartLid OL750UK Multicooker & Air Fryer - Stainless Steel &        
  Black",                                                                                                          
      "Price": "£339.99"                                                                                           
  })                                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'Product Name': 'NINJA Foodi MAX 15-in-1 SmartLid OL750UK Multicooker & Air Fryer - Stainless Steel 
& Black', 'Price': '£339.99'}

Captured a browser screenshot: (500, 806) pixels


[Step 5: Duration 1.87 seconds| Input tokens: 19,333 | Output tokens: 658]

Final output:
{'Product Name': 'NINJA Foodi MAX 15-in-1 SmartLid OL750UK Multicooker & Air Fryer - Stainless Steel & Black', 'Price': '£339.99'}


github_request = """Can you navigate to the profile of the most trending project, tell me the number of commits they have in the past year?"""

agent_output = agent.run(github_request +helium_instructions)
print("Final output:")
print(agent_output)

In [20]:
github_request = """Can you navigate to the profile of the most trending project, tell me the number of commits they have in the past year?"""

agent_output = agent.run(github_request +helium_instructions)
print("Final output:")
print(agent_output)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Can you navigate to the profile of the most trending project, tell me the number of commits they have in the    │
│ past year?                                                                                                      │
│ You can use helium to access websites. Don't bother about the helium driver, it's already managed.              │
│ We've already ran "from helium import *"                                                                        │
│ Then you can go to pages!                                                                                       │
│ Code:                                                                                                           │
│ ```py                                                                                                           │
│ go_to('github.com/trending')                                                                                    │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ You can directly click clickable elements by inputting the text that appears on them.                           │
│ Code:                                                                                                           │
│ ```py                                                                                                           │
│ click("Top products")                                                                                           │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ If it's a link:                                                                                                 │
│ Code:                                                                                                           │
│ ```py                                                                                                           │
│ click(Link("Top products"))                                                                                     │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ If you try to interact with an element and it's not found, you'll get a LookupError.                            │
│ In general stop your action after each button click to see what happens on your screenshot.                     │
│ Never try to login in a page.                                                                                   │
│                                                                                                                 │
│ To scroll up or down, use scroll_down or scroll_up with as an argument the number of pixels to scroll from.     │
│ Code:                                                                                                           │
│ ```py                                                                                                           │
│ scroll_down(num_pixels=1200) # This will scroll one viewport down                                               │
│ ```<end_code>                                                                                                   │
│                                                                                                                 │
│ When you have pop-ups with a cross icon to close, don'

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
To find the number of commits for the most trending project on GitHub, I will follow these steps:                  
                                                                                                                   
1. Navigate to the GitHub Trending page.                                                                           
2. Identify and click on the most trending project to go to its profile.                                           
3. Locate the section of the project page that provides commit statistics.                                         
4. Extract the number of commits in the past year.                                                                 
                                                                                                                   
Let's start by navigating to the trending projects page.                                                           
                                                                                                                   
<code>                                                                                                             
go_to('https://github.com/trending')                                                                               
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  go_to('https://github.com/trending')                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out: None

...

[Step 15: Duration 1.61 seconds| Input tokens: 64,289 | Output tokens: 840]

Reached max steps.

Captured a browser screenshot: (500, 806) pixels


[Step 16: Duration 3.18 seconds| Input tokens: 68,596 | Output tokens: 1,089]

Final output:
### Thoughts

To find the number of commits for the most trending project in the last year using Helium, follow these steps:

1. Navigate to the GitHub Trending page.
2. Click on the most trending project.
3. Check if there is a filter option to view commits for the past year.

### Code

```python
# Navigate to the GitHub Trending page
go_to('https://github.com/trending')

# Click on the most trending project
click("boykopovar / AnyPS5")

# Try accessing the commits page for potential filters
go_to("https://github.com/boykopovar/AnyPS5/commits")

# Check for any UI elements that allow filtering commits by time (e.g., past year)
click("All time")  # If filtering options are available

# Attempts to extract or filter the date may require hovering over date elements or filtering if present
```

### Notes

- If direct filtering isn't available via Helium, consider using the GitHub API to retrieve commit statistics.
- If manual inspection doesn't show the results, automating w